# Stationary
- A series is stationary when its statistical behaviour doesn’t change over time. In practice (called weak stationarity), three things must hold:

**Constant mean**: the series hovers around the same level.
**Constant variance**: the size of the wiggles stays about the same.
Time-independent relationships: how a value relates to the value k steps earlier depends only on k, not on when you look.

# Why does it matter?
- Models learn rules from the past and apply them to the future. If the mean and variance keep changing, the rules learned in training no longer hold later.
- ARIMA assumes it. The “I” in ARIMA (integrated) is literally the differencing step that makes the data stationary.
Fake relationships. Two unrelated trending series (say, ice cream sales and the price of AAPL, both rising) look strongly correlated. This is called spurious correlation.
- Forecast behaviour. Forecasts for stationary series settle back toward the mean. For non-stationary ones they can wander off.

#  Strict (Strong) Stationarity
- A time series is strictly stationary if the joint probability distribution of any collection of observations is completely invariant under time shifts.
- Condition: For any set of time points 

- What it means: All statistical moments (mean, variance, skewness, kurtosis, etc.) remain constant across time.
- In practice: Extremely rare and difficult to observe or verify in real-world data.

#  Weak (Covariance / Wide-Sense / Second-Order) Stationarity
This is the standard definition used in almost all practical ML and TSA workflows. Rather than requiring the entire distribution to be invariant, it only requires the first two statistical moments to be stable.
A process is weakly stationary if it satisfies three conditions:

**Constant Mean**: 



# Testing for stationary

#  Statistical tests: ADF and KPSS

- Eyes can fool you, so we use two formal tests. They ask opposite questions. This is the part people mix up.

Test	Null hypothesis (H0)	Small p-value (< 0.05) means
**ADF** (Augmented Dickey-Fuller)	The series has a unit root (non-stationary)	Reject H0, so it is stationary
**KPSS**	The series is stationary	Reject H0, so it is non-stationary

- ADF: small p = good (stationary). KPSS: small p = bad (non-stationary).

In [1]:
import warnings
from statsmodels.tsa.stattools import adfuller, kpss

def check_stationarity(series, name="series"):
    s = series.dropna()

    adf_stat, adf_p, *_ = adfuller(s, autolag="AIC")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")      # KPSS prints a warning when p is outside its table
        kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")

    adf_ok  = adf_p < 0.05       # ADF: small p  -> stationary
    kpss_ok = kpss_p >= 0.05     # KPSS: small p -> NOT stationary

    print(f"--- {name} ---")
    print(f"ADF  p-value: {adf_p:.4f} -> {'stationary' if adf_ok else 'NOT stationary'}")
    print(f"KPSS p-value: {kpss_p:.4f} -> {'stationary' if kpss_ok else 'NOT stationary'}")

    if adf_ok and kpss_ok:
        verdict = "STATIONARY"
    elif not adf_ok and not kpss_ok:
        verdict = "NOT stationary -> try differencing"
    elif kpss_ok and not adf_ok:
        verdict = "trend-stationary -> remove the trend"
    else:
        verdict = "difference-stationary -> use differencing"
    print("Verdict:", verdict, "\n")

check_stationarity(close, "AAPL Close price")

NameError: name 'close' is not defined